# Fine-Tuning Transformers with PyTorch and Hugging Face — worked study copy

Answers prepared for revision with AI assistance. IBM / Skills Network source notices and embedded instructor explanations are retained. This is not a course submission.

Code completion and execution are tracked separately in the [solution progress report](../../../docs/SOLUTION_PROGRESS.md). Training may require a separate environment, external data, compute or provider credentials.

[Original reference](../../../courses/10-fine-tuning-transformers/notebooks/fine-tuning-transformers-with-pytorch-and-hugging-face.ipynb) · [Course notes](../../../courses/10-fine-tuning-transformers/README.md)

**BOUNDED SMOKE RUN:** seed 42; CSVs sampled to at most 5,000 rows; one grid setting with two folds; training budgets capped; DataLoaders limited to 12 batches. These outputs are execution checks, not full-run model evaluations. Guard fp16 with CUDA availability and bound SFTTrainer to 2 steps on 8 rows for CPU smoke execution. Force CPU execution on macOS because transformers 4.45.2 with torch 2.2.2 raises IndexError in isin_mps_friendly on MPS.

In [1]:
import os, itertools, random
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
import numpy as np
np.random.seed(42)
random.seed(42)
import pandas as pd
_read_csv = pd.read_csv
def _bounded_csv(*args, **kwargs):
    data = _read_csv(*args, **kwargs)
    return data.sample(n=5000,random_state=42) if len(data)>5000 else data
pd.read_csv = _bounded_csv
from sklearn.model_selection import GridSearchCV
_grid_fit = GridSearchCV.fit
def _bounded_grid(self, *args, **kwargs):
    grids = self.param_grid if isinstance(self.param_grid,list) else [self.param_grid]
    self.param_grid = [{k:list(v)[:1] for k,v in grid.items()} for grid in grids[:1]]
    self.cv = 2
    self.n_jobs = 1
    self.verbose = 0
    return _grid_fit(self,*args,**kwargs)
GridSearchCV.fit = _bounded_grid
try:
    import torch
except ImportError:
    torch = None
if torch is not None:
    torch.manual_seed(42)
    torch.set_num_threads(1)
    _loader_iter = torch.utils.data.DataLoader.__iter__
    def _bounded_loader(self):
        return itertools.islice(_loader_iter(self),12)
    torch.utils.data.DataLoader.__iter__ = _bounded_loader


<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Fine-Tuning Transformers with PyTorch and Hugging Face**

Estimated time needed: **45** minutes


# Introduction

This project aims to introduce you to the process of loading and fine-tuning pretrained large language models (LLMs) 

You will learn how to implement the training loop of a model using pytorch to tune a model on task-specific data, as well as fine-tuning a model on task-specific data using the SFTTrainer module from Hugging Face. Finally, you will learn how to evaluate the performance of the fine-tuned models.

By the end of this project, you will have a solid understanding of how to leverage pretrained LLMs and fine-tune them for your specific use cases, empowering you to create powerful and customized natural language processing solutions.


# __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Importing-required-libraries">Importing required libraries</a></li>
        </ol>
    </li>
    <li><a href="#Supervised-Fine-tuning-with-PyTorch)">Supervised Fine-tuning with Pytorch</a>
        <ol>
            <li><a href="#Dataset-preparations">Dataset preparations</a></li>
            <li><a href="#Train-the-model">Train the model</a></li>
            <li><a href="#Evaluate">Evaluate</a></li>
            <li><a href="#Loading-the-saved-model">Loading the saved model</a></li>
        </ol>
    </li>
    <li><a href="#Exercise:-Training-a-conversational-model-using-SFTTrainer">Exercise: Training a conversational model using SFTTrainer</a></li>
</ol>


---


# Objectives

After completing this lab, you will be able to:

 - Load pretrained LLMs from Hugging Face and make inferences
 - Fine-tune a model on task-specific data using the SFTTrainer module from Hugging Face
 - Load a SFTTrainer pretrained model and make comparisons
 - Evaluate the model


---


# Setup


### Installing required libraries

The following required libraries are pre-installed in the Skills Network Labs environment. However, if you run this notebook commands in a different Jupyter environment (e.g. Watson Studio or Ananconda), you will need to install these libraries by removing the `#` sign before `!pip` in the code cell below.


### Importing required libraries

_It is recommended that you import all required libraries in one place (here):_
* Note: If you get an error after running the cell below, try restarting the Kernel, as some packages need a restart to be effective.


In [2]:
import torch
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False
from torchmetrics import Accuracy
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForSequenceClassification, BertConfig, BertForMaskedLM, TrainingArguments, Trainer, TrainingArguments
from transformers import AutoTokenizer, BertTokenizerFast, TextDataset, DataCollatorForLanguageModeling
from transformers import pipeline
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from tqdm.auto import tqdm
import math
import time
import matplotlib.pyplot as plt

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


<repository>/.venv-legacy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---


# Supervised Fine-tuning with Pytorch

Fine-tuning Transformers, specifically BERT (Bidirectional Encoder Representations from Transformers), refers to the process of training a pretrained BERT model on a specific downstream task. BERT is an encoder-only language model that has been pretrained on a large corpus of text to learn contextual representations of words.

Fine-tuning BERT involves taking the pretrained model and further training it on a task-specific dataset, such as sentiment analysis or question answering. During fine-tuning, the parameters of the pretrained BERT model are updated and adapted to the specifics of the target task.

This process is important because it allows you to leverage the knowledge and language understanding captured by BERT and apply it to different tasks. By fine-tuning BERT, you can benefit from its contextual understanding of language and transfer that knowledge to specific domain-specific or task-specific problems. Fine-tuning enables BERT to learn from a smaller labeled dataset and generalize well to unseen examples, making it a powerful tool for various natural language processing tasks. It helps to bridge the gap between pretraining on a large corpus and the specific requirements of downstream applications, ultimately improving the performance and effectiveness of models in various real-world scenarios.


## Dataset preparations

The Yelp review dataset is a widely used dataset in natural language processing (NLP) and sentiment analysis research. It consists of user reviews and accompanying metadata from the Yelp platform, which is a popular online platform for reviewing and rating local businesses such as restaurants, hotels, and shops.

The dataset includes 6,990,280 reviews written by Yelp users, covering a wide range of businesses and locations. Each review typically contains the text of the review itself alongwith the star rating given by the user (ranging from 1 to 5).

Our aim in this lab, is to fine-tune a pretrained BERT model to predict the ratings from reviews.


Let's start by loading the yelp_review data:


In [3]:
dataset = load_dataset('yelp_review_full')
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

Let's check a sample record of the dataset:


In [4]:
dataset['train'][100]

{'label': 0,
 'text': 'My expectations for McDonalds are t rarely high. But for one to still fail so spectacularly...that takes something special!\\nThe cashier took my friends\'s order, then promptly ignored me. I had to force myself in front of a cashier who opened his register to wait on the person BEHIND me. I waited over five minutes for a gigantic order that included precisely one kid\'s meal. After watching two people who ordered after me be handed their food, I asked where mine was. The manager started yelling at the cashiers for \\"serving off their orders\\" when they didn\'t have their food. But neither cashier was anywhere near those controls, and the manager was the one serving food to customers and clearing the boards.\\nThe manager was rude when giving me my order. She didn\'t make sure that I had everything ON MY RECEIPT, and never even had the decency to apologize that I felt I was getting poor service.\\nI\'ve eaten at various McDonalds restaurants for over 30 years. 

the label is the key of the class label


In [5]:
dataset['train'][100]['label']

0

there is also the text


In [6]:
dataset['train'][100]['text']

'My expectations for McDonalds are t rarely high. But for one to still fail so spectacularly...that takes something special!\\nThe cashier took my friends\'s order, then promptly ignored me. I had to force myself in front of a cashier who opened his register to wait on the person BEHIND me. I waited over five minutes for a gigantic order that included precisely one kid\'s meal. After watching two people who ordered after me be handed their food, I asked where mine was. The manager started yelling at the cashiers for \\"serving off their orders\\" when they didn\'t have their food. But neither cashier was anywhere near those controls, and the manager was the one serving food to customers and clearing the boards.\\nThe manager was rude when giving me my order. She didn\'t make sure that I had everything ON MY RECEIPT, and never even had the decency to apologize that I felt I was getting poor service.\\nI\'ve eaten at various McDonalds restaurants for over 30 years. I\'ve worked at more t

You can select a portion of data to decrease the training time:


In [7]:
dataset['train'] = dataset['train'].select([i for i in range(1000)])
dataset['test'] = dataset['test'].select([i for i in range(200)])

There are two data fields:
- label: the label for the review
- text: a string containing the body of the user review


### Tokenizing data

The next step is to load a BERT tokenizer to tokenize, pad and truncate reviews to handle variable-length sequences:


In [8]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

The keys in each element of tokenized_datasets are 'label', 'text', 'input_ids', 'token_type_ids', and 'attention_mask'.


In [9]:
tokenized_datasets['train'][0].keys()

dict_keys(['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'])

To apply the preprocessing function over the entire dataset, let's use the map method. You can speed up the map function by setting batched=True to process multiple elements of the dataset at once:


Since the model is built on the PyTorch framework, it is crucial to prepare the dataset in a format that PyTorch can readily process. Follow these steps to ensure compatibility:


In [10]:
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch')

the result is a set of tensors with the keys as:  'labels', 'input_ids', 'token_type_ids', 'attention_mask'


In [11]:
tokenized_datasets['train'][0].keys()

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

### DataLoader


Next, create a DataLoader for train and test datasets so you can iterate over batches of data:


In [12]:
train_dataloader = DataLoader(tokenized_datasets['train'], shuffle=True, batch_size=2)
eval_dataloader = DataLoader(tokenized_datasets['test'], batch_size=2)

## Train the model


You’re ready to start training your model, now!
In this section, you will learn to create the training loop from scratch without the help of the Hugging Face trainer class.
In the MLM task, you utilized the Hugging Face trainer module. Now, you will develop the trainer yourself.


### Load a pretrained model


Here, you'll load a pretrained classification model with 5 classes:


In [13]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-cased', num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Optimizer and learning rate schedule

Let's create an optimizer and learning rate scheduler to fine-tune the model. You can use the AdamW optimizer from PyTorch:


In [14]:
optimizer = AdamW(model.parameters(), lr=0.0005)
num_epochs = 2
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = LambdaLR(optimizer, lr_lambda=lambda current_step: 1 - current_step / num_training_steps)

 Check if CUDA is available and, then set the device accordingly.


In [15]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

### Training loop

You are ready to fine-tune the model. To keep track of training progress, let's use the "tqdm" library to add a progress bar over the number of training steps.
The train_model function trains a model using a set of training data provided through a dataloader. It begins by setting up a progress bar to help monitor the training progress visually. The model is switched to training mode, which is necessary for certain model behaviors like dropout to work correctly during training. The function processes the data in batches for each epoch, which involves several steps for each batch: transferring the data to the correct device (like a GPU), running the data through the model to get outputs and calculate loss, updating the model's parameters using the calculated gradients, adjusting the learning rate, and clearing the old gradients. These steps are repeated for each batch of data, and the progress bar is updated accordingly to reflect the progress. Once all epochs are completed, the trained model is saved to be used later.









In [16]:
def train_model(model, tr_dataloader):
    progress_bar = tqdm(range(num_training_steps))
    model.train()
    tr_losses = []
    for epoch in itertools.islice(range(num_epochs), 2):
        total_loss = 0
        for batch in tr_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            total_loss += loss.item()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)
        tr_losses.append(total_loss / len(tr_dataloader))
    plt.plot(tr_losses)
    plt.title('Training loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.show()

## Evaluate
The evaluate_model function works similarly to the train_model function but is used for evaluating the model's performance instead of training it. It uses a dataloader to process data in batches, setting the model to evaluation mode to ensure accuracy in measurements and disabling gradient calculations since it's not training. The function calculates predictions for each batch, updates an accuracy metric, and finally, prints the overall accuracy after processing all batches.








In [17]:
def evaluate_model(model, evl_dataloader):
    metric = Accuracy(task='multiclass', num_classes=5).to(device)
    model.eval()
    with torch.no_grad():
        for batch in evl_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            metric(predictions, batch['labels'])
    accuracy = metric.compute()
    print('Accuracy:', accuracy.item())

You can now train the model. This process will take a long time, and it is highly recommended that you do this only if you have the required resources. Please uncomment the following code to train the model.


![loss_gpt.png](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/HausLW2F_w30s1UK0zj7mQ/training-loss-BERT-Classification.png)


You are now ready to learn how to tune a more complex model that can generate conversations between a human and an assistant.


## Loading the saved model
If you want to skip training and load the model that you trained for 10 epochs, go ahead and uncomment the following cell:


In [18]:
from pathlib import Path as _LabPath
from urllib.request import urlopen as _lab_urlopen
from shutil import copyfileobj as _lab_copyfileobj

def _download_lab_asset(url, filename):
    target = _LabPath(filename)
    if not target.is_file():
        partial = target.with_name(target.name + '.part')
        try:
            with _lab_urlopen(url, timeout=60) as response, partial.open('wb') as output:
                _lab_copyfileobj(response, output)
            partial.replace(target)
        finally:
            partial.unlink(missing_ok=True)
    return target
_download_lab_asset('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/wFhKpkBMSgjmZKRSyayvsQ/bert-classification-model.pt', 'bert-classification-model.pt')
model.load_state_dict(torch.load('bert-classification-model.pt', map_location=torch.device('cpu')))

<All keys matched successfully>

You can now evaluate the model. Please note that this process will take a while.


In [19]:
evaluate_model(model, eval_dataloader)

Accuracy: 0.2083333283662796


You are now ready to learn to tune a more complex model that can generate conversations between a human and an assistant using SFTtrainer.


# Exercise: Training a conversational model using SFTTrainer

The SFTTrainer from the trl (Transformers Reinforcement Learning) library is a tool used for supervised fine-tuning of language models. It helps refine pre-trained models using specific datasets to enhance their performance on targeted tasks.


## Objective
Explore how fine-tuning a decoder transformer using a specific dataset affects the quality of the generated responses in a question-answering task.


Step 1- Load the train split of "timdettmers/openassistant-guanaco" dataset from Hugging Face:


In [20]:
dataset = load_dataset('timdettmers/openassistant-guanaco', split='train')
dataset[0]

Repo card metadata block was not found. Setting CardData to empty.


{'text': '### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced bargaining po

<details>
    <summary>Click here for Solution</summary>

```python
# load dataset
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")
dataset[0]
```

</details>


Step 2- Load the pretrained causal model "facebook/opt-350m" along with its tokenizer from Hugging Face:


In [21]:
model = AutoModelForCausalLM.from_pretrained('facebook/opt-350m')
tokenizer = AutoTokenizer.from_pretrained('facebook/opt-350m')

<details>
    <summary>Click here for Solution</summary>

```python
# load Hugging Face pretrained model and tokenizer
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")
```

</details>


Step 3- Create instruction and response templates based on the train dataset format:


In [22]:
instruction_template = '### Human:'
response_template = '### Assistant:'

<details>
    <summary>Click here for Solution</summary>

```python
instruction_template = "### Human:"
response_template = "### Assistant:"
```

</details>


Step 4- Create a collator to curate data in the appropriate shape for training using **"DataCollatorForCompletionOnlyLM"**:


In [23]:
collator = DataCollatorForCompletionOnlyLM(instruction_template=instruction_template, response_template=response_template, tokenizer=tokenizer, mlm=False)

<details>
    <summary>Click here for Solution</summary>

```python
collator = DataCollatorForCompletionOnlyLM(instruction_template=instruction_template, response_template=response_template, tokenizer=tokenizer, mlm=False)

```

</details>


Step 5- Create an SFTTrainer object and pass the model as well as the dataset and collator: 


In [24]:
training_args = SFTConfig(output_dir='/tmp', num_train_epochs=1, max_steps=2, save_strategy='no', fp16=torch.cuda.is_available(), per_device_train_batch_size=2, per_device_eval_batch_size=2, max_seq_length=256, do_eval=False)
trainer = SFTTrainer(model, args=training_args, train_dataset=dataset.select(range(min(8, len(dataset)))), dataset_text_field='text', data_collator=collator)

max_steps is given, it will override any value given in num_train_epochs


<details>
    <summary>Click here for Solution</summary>

```python

training_args = SFTConfig(
    output_dir="/tmp",
    num_train_epochs=10,
    #learning_rate=2e-5,
    save_strategy="epoch",
    fp16=True,
    per_device_train_batch_size=2,  # Reduce batch size
    per_device_eval_batch_size=2,  # Reduce batch size
    #gradient_accumulation_steps=4,  # Accumulate gradients
    max_seq_length=1024,
    do_eval=True
)

trainer = SFTTrainer(
    model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text",
    data_collator=collator,
)
```

</details>


Step 6- Prompt the pretrained model with a specific question: 


In [25]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=70)
print(pipe('Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.')[0]['generated_text'])

Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.

The term "monopsony" is used in the context of the "mono" (mono-economy) model. The term "mono-economy" is used in the context of the "mono-economy" model. The term "mono-economy" is used in the context of


<details>
    <summary>Click here for Solution</summary>

```python
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer,max_new_tokens=70)
print(pipe('''Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.''')[0]["generated_text"])
```

</details>


Looks like the model is barely aware of what "monopsony" is in the context of economics.


Step 6A (Optional)- Train the model:


In [26]:
trainer.train()

  0%|                                                    | 0/2 [00:00<?, ?it/s]

 50%|██████████████████████                      | 1/2 [00:04<00:04,  4.01s/it]

100%|████████████████████████████████████████████| 2/2 [00:06<00:00,  2.99s/it]

100%|████████████████████████████████████████████| 2/2 [00:06<00:00,  2.99s/it]

100%|████████████████████████████████████████████| 2/2 [00:06<00:00,  3.15s/it]

{'train_runtime': 6.2877, 'train_samples_per_second': 0.636, 'train_steps_per_second': 0.318, 'train_loss': 2.109220266342163, 'epoch': 0.5}


TrainOutput(global_step=2, training_loss=2.109220266342163, metrics={'train_runtime': 6.2877, 'train_samples_per_second': 0.636, 'train_steps_per_second': 0.318, 'total_flos': 1863831257088.0, 'train_loss': 2.109220266342163, 'epoch': 0.5})

<details>
    <summary>Click here for Solution</summary>

```python
trainer.train()
```

</details>


If you do not have enough resources to run the training, load the tuned model we provide here: "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Assistant_model.pt":



Step 6B- Load the tuned model: 


In [27]:
from pathlib import Path as _LabPath
from urllib.request import urlopen as _lab_urlopen
from shutil import copyfileobj as _lab_copyfileobj

def _download_lab_asset(url, filename):
    target = _LabPath(filename)
    if not target.is_file():
        partial = target.with_name(target.name + '.part')
        try:
            with _lab_urlopen(url, timeout=60) as response, partial.open('wb') as output:
                _lab_copyfileobj(response, output)
            partial.replace(target)
        finally:
            partial.unlink(missing_ok=True)
    return target
_download_lab_asset('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Assistant_model.pt', 'Assistant_model.pt')
model.load_state_dict(torch.load('Assistant_model.pt', map_location=torch.device('cpu')))

<All keys matched successfully>

<details>
    <summary>Click here for Solution</summary>

```python
!wget 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Assistant_model.pt'
model.load_state_dict(torch.load('Assistant_model.pt',map_location=torch.device('cpu')))
```

</details>


Step 7- Check how the tuned model performs in answering the same specialized question:


In [28]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=70)
print(pipe('Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.')[0]['generated_text'])

Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.

The term "monopsony" is often used to refer to the use of monopsonies in the labor market, where companies use their power to control the market and manipulate prices. However, the term "monopsony" has a broader meaning in economics, as it refers to the use of companies to manipulate prices and manipulate


<details>
    <summary>Click here for Solution</summary>

```python
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer,max_new_tokens=70)
print(pipe('''Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.''')[0]["generated_text"])
```

</details>


# Congratulations! You have completed the lab


## Authors


[Fateme Akbari](https://author.skills.network/instructors/fateme_akbari) is a Ph.D. candidate in Information Systems at McMaster University with demonstrated research experience in Machine Learning and NLP.


© Copyright IBM Corporation. All rights reserved.
